<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l3.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L3 · Estrategia neutral a beta
Demean por era y residuo contra beta.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l3.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l3.csv'), Path('data/c6_l3.csv'), Path('c6_l3.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))


In [ ]:
# Diagnóstico: ¿cuánta beta carga la predicción?
carga = df['pred'].corr(df['beta'])
print(f'Correlación pred vs beta (antes)={carga:.4f}')
assert np.isfinite(carga)


In [ ]:
# Neutralización por era: demean + residuo de regresión vs beta
def neutralizar(g):
    d = g['pred'] - g['pred'].mean()
    b = g['beta'].values
    coef = np.polyfit(b, d.values, 1)[0]
    return d - coef * b
df['neutral'] = df.groupby('era', group_keys=False).apply(lambda g: pd.Series(neutralizar(g), index=g.index))
carga_n = df['neutral'].corr(df['beta'])
print(f'Correlación neutral vs beta (después)={carga_n:.4f}')
assert abs(carga_n) < 0.3, 'sigue cargada de beta'


In [ ]:
# ¿Sobrevive el filo? CORR por era antes vs después
c_antes = df.groupby('era').apply(lambda g: g['pred'].rank().corr(g['ret_futuro'].rank()), include_groups=False).mean()
c_desp = df.groupby('era').apply(lambda g: g['neutral'].rank().corr(g['ret_futuro'].rank()), include_groups=False).mean()
print(f'CORR antes={c_antes:.4f}  CORR neutral={c_desp:.4f}')
assert np.isfinite(c_antes) and np.isfinite(c_desp)
assert c_desp > 0, 'el filo de selección debe sobrevivir a la neutralización'



In [ ]:
# Chequeo automático L3
assert abs(df['neutral'].corr(df['beta'])) < 0.3
assert df['neutral'].notna().all()
print('OK L3: señal neutral a beta verificada')
